# dfthcsos starter: rebuild, verify and submit

This notebook walks through one full loop on the `dfthcsos` task with the dfthcsos starter kit:

1. explore how much angle precision the reference factorization needs,
2. rebuild the ANGLE circuits with a small change (a toy edit),
3. score the result with the competition's own verifier, next to the reference submission's circuits,
4. package a complete submission: your new `dfthcsos/` folder plus the sample `invsqrt` and `dqi` circuits.

It takes about 10 minutes on a CPU session and needs internet access, to install Qualtran.

## Setup

The notebook mounts the competition data and the starter kit dataset. Qualtran, which the starter kit uses to build its circuits, is installed below.

- `verifiers/`: the problem spec, the verifier for each task, with `dfthcsos`'s inputs (`FeMoCo76o113e.fcidump` and the fixed symmetry shifts),
- the starter kit dataset's `starter_kit/`: the dfthcsos starter kit,
- `kickmix/`: `kickmix.so`, the circuit simulator, built for this notebook image.

Everything runs straight from the read-only data; only the final `submission.zip` is written to `/kaggle/working`.

In [ ]:
# %pip install --upgrade cirq-core pandas
# %pip install -q "qualtran @ git+https://github.com/quantumlib/Qualtran.git@c58703d00b01c346d3e77d3913cf2d6e9e556be3"
# %pip install kickmix


^C
Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\kuijp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [28]:

import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

COMPETITION = Path('kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
VERIFIERS = COMPETITION / 'verifiers'
KIT = Path('kaggle/input/verifiable-circuit-compilation-starter-kit/starter_kit')
KICKMIX = COMPETITION / 'kickmix'
BUILD = Path('./tmp_dfthcsos_build')

# kickmix, the verifiers (dfthcsos, dqi, invsqrt) and the starter kit (dfthc, starter, verifier),
# for this kernel and for the scripts run below.
paths = [str(KICKMIX), str(VERIFIERS), str(KIT)]
sys.path[:0] = paths
os.environ['PYTHONPATH'] = os.pathsep.join([*paths, os.environ.get('PYTHONPATH', '')])

import kickmix

print('kickmix loaded from', kickmix.__file__)
print('starter kit at', KIT)

kickmix loaded from C:\Users\kuijp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\kickmix.cp312-win_amd64.pyd
starter kit at kaggle\input\verifiable-circuit-compilation-starter-kit\starter_kit


## 1. How much precision do the angles need?

The verifier rotates each vector with angles stored as `brot`-bit words. Fewer bits mean narrower circuits, but the
rounded words must still reproduce the Hamiltonian within 20 mHa. The kit's `encode.py` scores the reference
factorization at several precisions without building any circuits:

In [24]:
!python kaggle/input/verifiable-circuit-compilation-starter-kit/starter_kit/starter/task2_angles/encode.py FeMoCo76o113e --tree log_depth --rounding nearest --brots 11 12 13


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\kuijp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
FeMoCo76o113e: FeMoCo76o113e_65_60_60_ep50000_seed0.npz, tree=log_depth, rounding=nearest
  brot  error_1 (Ha)  error_2 (mHa)     Lambda     E_gap  lambda_eff      score
 exact     6.973e-15        16.1332   308.4485   3.95802    49.25469   49.25469
    11     6.480e-03        25.0354   308.4587   3.96828    49.31889        inf
    12     3.185e-03        18.6396   308.4559   3.96548    49.30137   49.30137
    13     1.593e-03        16.7973   308.4471   3.95664    49.24606   49.24606


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

At `brot = 11` the two-body error (`error_2`) exceeds 20 mHa, so the score is `inf`: 12 bits is the least this
factorization can use. Reaching lower precision means changing the factorization itself (Part 1 of the kit).

## 2. Rebuild the circuits with a toy edit

Each ANGLE circuit loads a batch of `J` angle words, so its width is about `J * brot` qubits plus the address and some
workspace, and every circuit must fit in 128 qubits. The reference submission uses `J = 8`; at `brot = 12`, `J = 9`
would need 131 qubits and be rejected. As a toy edit we use `J = 7`: narrower circuits, but one more of them.

The kit's `build_circuits.py` builds the family with Qualtran QROMs, simulates it on every address before writing
anything, and reports its score. It takes about 6 minutes:

In [25]:
!python {KIT}/starter/task3_circuits/build_circuits.py FeMoCo76o113e --tree log_depth --brot 12 --batch-size 7 --output {BUILD}

FeMoCo76o113e: wrote 11 circuits to tmp_dfthcsos_build
  k  qubits  max_magic  reaction_depth  instructions
  0     107       3974              11        167156
  1     107       3974              11        166677
  2     107       3974              11        166746
  3     107       3974              11        167222
  4     107       3974              11        182692
  5     107       3974              11        194777
  6     107       3974              11        194042
  7     107       3974              11        195161
  8     107       3974              11        195186
  9     107       3974              11        195064
 10     107       3974              11        194113
Toffoli = 101185.0 (2 * ANGLES 87428.0 + outer prep 846 + inner prep 9461 + Givens 3450)
qubits  = 718 (persistent 174 + shared temp 544; ANGLES width 107); ANGLES <= 128: ok
recovered words: error_1 = 3.1852 mHa, error_2 = 18.6396 mHa, lambda_eff = 49.301369
score = lambda_eff * Toffoli * qubits = 3.58179e+

## 3. Score it with the competition's verifier

`verifiers/dfthcsos/verify.py` is the verifier the leaderboard uses, here without the secret salt, so its simulated
shots differ from the leaderboard's. We score the toy edit and, for comparison, the reference circuits in the
sample submission. Lower cost is better.

In [31]:
def dfthcsos_cost(circuits: Path) -> float:
    """The competition verifier's dfthcsos cost for a folder of angle_k_* circuits and submission.npz."""
    # verify.py exits 1 when it rejects a submission; its JSON says why, which is raised below.
    result = subprocess.run([sys.executable, str(VERIFIERS / 'dfthcsos' / 'verify.py'), str(circuits)], capture_output=True, text=True, check=False)
    summary = json.loads(result.stdout)
    if summary.get('status') != 'PASS':
        raise RuntimeError(summary)
    return summary['score']

reference = dfthcsos_cost(COMPETITION/'sample_submission'/'dfthcsos')
toy_edit = dfthcsos_cost(BUILD)
print(f'reference: {reference:.4g}   toy edit: {toy_edit:.4g}   ratio: {reference / toy_edit:.4f}')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

The leaderboard score is the geometric mean, over the three tasks, of the reference cost divided by yours. With the
sample circuits for `invsqrt` and `dqi` (a ratio of 1 each), this submission would score about:

In [ ]:
print(f'{(reference / toy_edit) ** (1 / 3):.4f}')

0.9785


The toy edit is valid but slightly worse than the reference: the extra circuit costs more Toffolis than the narrower
word registers save.

## 4. Package a submission

A submission is a zip of `invsqrt.kmb` (or `.kmx`), `dqi.kmb` (or `.kmx`) and a `dfthcsos/` folder holding
`angle_k_0` ... `angle_k_<n-1>` and `submission.npz`. Here the `dfthcsos/` folder is the toy edit's:

In [ ]:
sample = COMPETITION / 'sample_submission'
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['invsqrt.kmb', 'dqi.kmb']:
        archive.write(sample / name, name)
    circuits = sorted(BUILD.glob('angle_k_*.kmx'), key=lambda path: int(path.stem.removeprefix('angle_k_')))
    for path in [*circuits, BUILD / 'submission.npz']:
        archive.write(path, f'dfthcsos/{path.name}')
print(*zipfile.ZipFile('/kaggle/working/submission.zip').namelist(), sep='\n')

invsqrt.kmb
dqi.kmb
dfthcsos/angle_k_0.kmx
dfthcsos/angle_k_1.kmx
dfthcsos/angle_k_2.kmx
dfthcsos/angle_k_3.kmx
dfthcsos/angle_k_4.kmx
dfthcsos/angle_k_5.kmx
dfthcsos/angle_k_6.kmx
dfthcsos/angle_k_7.kmx
dfthcsos/angle_k_8.kmx
dfthcsos/angle_k_9.kmx
dfthcsos/angle_k_10.kmx
dfthcsos/submission.npz


## Where to go from here

- **Part 1, factorization:** `starter/task1_factorization/train_df_thc_qaggle.py` trains a new factorization, seeded
  from a baseline. The verifier fixes the symmetry shift to the baseline's and requires the LCU normalization
  `Lambda > 49.0` and the SOS gap `E_gap > 3.914898` Ha (the baseline's are 308.46 and 3.965). The score's
  `lambda_eff` (49.30 for the baseline) has no floor of its own, so a smaller `Lambda` pays, as do factorization
  sizes `R, B, C` and precision that make the circuits cheaper.
- **Part 2, tree and precision:** `starter/task2_angles/` has the path and log-depth Givens trees and the rounding
  options.
- **Part 3, circuits:** the baseline loads each batch with an ordinary QROM. Any construction is allowed as long as the
  simulated circuits load the submitted words; `verifiers/dfthcsos/SCORING_README.md` has the full contract.
- **The other tasks:** `verifiers/invsqrt/` and `verifiers/dqi/` each have a reference-circuit script that writes
  the sample circuits, a starting point for your own.